# Causal Discovery Experiment

This notebook demonstrates the AEGIS causal discovery pipeline using DAG-GNN.

**Goals:**
1. Load sample tabular data
2. Run PC algorithm baseline (if available)
3. Run DAG-GNN discovery
4. Compare results (edge counts, accuracy)
5. Visualize the causal graph
6. Detect proxy chains
7. Summary and conclusions

In [ ]:
# Cell 1: Imports and setup
import sys
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

# Check for optional dependencies
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    HAS_TORCH = True
except ImportError:
    print("PyTorch: not installed (DAG-GNN will be unavailable)")
    HAS_TORCH = False

try:
    import networkx as nx
    print(f"NetworkX: {nx.__version__}")
    HAS_NX = True
except ImportError:
    print("NetworkX: not installed")
    HAS_NX = False

In [ ]:
# Cell 2: Load sample data
from sklearn.datasets import make_classification

# Generate synthetic data with known causal structure
n_samples = 500
n_features = 8
n_informative = 6

X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=n_informative,
    n_redundant=1,
    n_clusters_per_class=2,
    random_state=42,
)

# Add feature names
feature_names = [f"X{i}" for i in range(n_features)]

# Create a DataFrame for easier inspection
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f"Data shape: {df.shape}")
print(f"\nFeature statistics:")
print(df.describe().round(3))
print(f"\nTarget distribution: {np.bincount(y)}")
print(f"\nCorrelation with target:")
print(df.corr()['target'].sort_values(ascending=False).round(3))

In [ ]:
# Cell 3: Run PC algorithm for baseline
pc_edges = []
pc_available = False

try:
    from app.ml.causal.pc_algorithm import PCAlgorithm
    
    print("Running PC algorithm baseline...")
    t0 = time.time()
    
    pc = PCAlgorithm(alpha=0.05)
    pc_result = pc.run(X, feature_names)
    
    pc_time = time.time() - t0
    pc_available = True
    
    if hasattr(pc_result, 'edges'):
        pc_edges = pc_result.edges
    elif isinstance(pc_result, dict):
        pc_edges = pc_result.get('edges', [])
    elif isinstance(pc_result, list):
        pc_edges = pc_result
    
    print(f"PC algorithm completed in {pc_time:.2f}s")
    print(f"Edges found: {len(pc_edges)}")
    if pc_edges:
        print("Edge list:")
        for edge in pc_edges[:20]:
            print(f"  {edge}")
        if len(pc_edges) > 20:
            print(f"  ... and {len(pc_edges) - 20} more")
    
except ImportError as e:
    print(f"PC algorithm not available: {e}")
    print("Skipping PC algorithm baseline.")
except Exception as e:
    print(f"PC algorithm failed: {e}")
    print("Skipping PC algorithm baseline.")

pc_n_edges = len(pc_edges)

In [ ]:
# Cell 4: Run DAG-GNN discovery
dag_result = None

if HAS_TORCH:
    from app.ml.causal.dag_gnn import DAGGNNDiscovery
    
    print("Running DAG-GNN discovery...")
    print(f"  input_dim={n_features}, hidden_dim=64, latent_dim=16")
    print(f"  max_epochs=50 (reduced for demo), threshold=0.3")
    
    t0 = time.time()
    
    discovery = DAGGNNDiscovery(
        input_dim=n_features,
        hidden_dim=64,
        latent_dim=16,
        threshold=0.3,
        max_epochs=50,  # Reduced for demo
        patience=10,
    )
    
    dag_result = discovery.discover(X, feature_names=feature_names)
    dag_time = time.time() - t0
    
    print(f"\nDAG-GNN completed in {dag_time:.2f}s")
    print(f"  Nodes: {dag_result.num_nodes}")
    print(f"  Edges: {dag_result.num_edges}")
    print(f"  Sparsity: {dag_result.sparsity:.4f}")
    
    if dag_result.edge_list:
        print(f"\n  Edge list (sorted by weight):")
        for src, tgt, w in sorted(dag_result.edge_list, key=lambda e: -e[2]):
            print(f"    {src} -> {tgt}  (weight={w:.4f})")
    else:
        print("  No edges found above threshold.")
else:
    print("PyTorch not available. Skipping DAG-GNN discovery.")
    print("Install PyTorch to enable DAG-GNN: pip install torch")
    dag_time = None

In [ ]:
# Cell 5: Compare results
print("=" * 60)
print("Causal Discovery Comparison")
print("=" * 60)

dag_n_edges = dag_result.num_edges if dag_result else 0

print(f"\n{'Method':<20} {'Edges':>8} {'Time (s)':>10}")
print("-" * 40)
if pc_available:
    print(f"{'PC Algorithm':<20} {pc_n_edges:>8} {pc_time:>10.2f}")
else:
    print(f"{'PC Algorithm':<20} {'N/A':>8} {'N/A':>10}")

if dag_result:
    print(f"{'DAG-GNN':<20} {dag_n_edges:>8} {dag_time:>10.2f}")
else:
    print(f"{'DAG-GNN':<20} {'N/A':>8} {'N/A':>10}")

print()

# Compare edge overlap if both methods succeeded
if pc_available and dag_result:
    dag_edge_set = set((s, t) for s, t, _ in dag_result.edge_list)
    
    # Parse PC edges into a set
    pc_edge_set = set()
    for edge in pc_edges:
        if isinstance(edge, tuple) and len(edge) >= 2:
            pc_edge_set.add((str(edge[0]), str(edge[1])))
    
    overlap = dag_edge_set & pc_edge_set
    dag_only = dag_edge_set - pc_edge_set
    pc_only = pc_edge_set - dag_edge_set
    
    if dag_edge_set or pc_edge_set:
        jaccard = len(overlap) / max(len(dag_edge_set | pc_edge_set), 1)
    else:
        jaccard = 0.0
    
    print(f"Edge overlap analysis:")
    print(f"  DAG-GNN edges:  {len(dag_edge_set)}")
    print(f"  PC edges:        {len(pc_edge_set)}")
    print(f"  Overlap:         {len(overlap)}")
    print(f"  DAG-GNN only:    {len(dag_only)}")
    print(f"  PC only:         {len(pc_only)}")
    print(f"  Jaccard index:   {jaccard:.4f}")
    
    if overlap:
        print(f"\n  Shared edges:")
        for edge in sorted(overlap):
            print(f"    {edge[0]} -> {edge[1]}")

In [ ]:
# Cell 6: Visualize causal graph
if HAS_NX and dag_result and dag_result.graph is not None:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    G = dag_result.graph
    
    # Layout
    try:
        pos = nx.spring_layout(G, seed=42, k=2.0)
    except Exception:
        pos = nx.circular_layout(G)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                          node_size=1500, alpha=0.9, ax=ax)
    
    # Draw edges with width proportional to weight
    if G.number_of_edges() > 0:
        edges = G.edges(data=True)
        weights = [d.get('weight', 0.5) for _, _, d in edges]
        max_w = max(weights) if weights else 1
        widths = [2 + 6 * (w / max(max_w, 1e-8)) for w in weights]
        
        nx.draw_networkx_edges(G, pos, edge_color='steelblue',
                               width=widths, alpha=0.7, ax=ax,
                               arrows=True, arrowsize=20,
                               connectionstyle='arc3,rad=0.1')
    
    # Labels
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', ax=ax)
    
    # Edge weight labels
    edge_labels = {}
    for u, v, d in G.edges(data=True):
        w = d.get('weight', 0)
        edge_labels[(u, v)] = f"{w:.3f}"
    
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                      font_size=8, ax=ax)
    
    ax.set_title(f"DAG-GNN Causal Graph\n{dag_result.num_nodes} nodes, "
                 f"{dag_result.num_edges} edges, sparsity={dag_result.sparsity:.3f}",
                 fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print("Graph visualization rendered above.")
    
    # Additional: degree analysis
    print(f"\nNode degree analysis:")
    print(f"  {'Node':<10} {'In-degree':>10} {'Out-degree':>12}")
    print(f"  {'-'*34}")
    for node in sorted(G.nodes()):
        in_d = G.in_degree(node)
        out_d = G.out_degree(node)
        print(f"  {node:<10} {in_d:>10} {out_d:>12}")

else:
    print("Graph visualization skipped (requires networkx and dag_result).")
    if not HAS_NX:
        print("  Install networkx: pip install networkx")
    if not dag_result:
        print("  DAG-GNN did not produce a result.")

In [ ]:
# Cell 7: Detect proxy chains
proxy_chains = []

if HAS_NX and dag_result and dag_result.graph is not None:
    try:
        from app.ml.causal.proxy_chain_detector import ProxyChainDetector
        
        detector = ProxyChainDetector(max_chain_length=3)
        result = detector.detect(
            graph=dag_result.graph,
            sensitive_nodes=None,  # Auto-detect
            outcome_node=None,      # Auto-detect
        )
        
        proxy_chains = result.chains
        
        print(f"Proxy Chain Detection Results:")
        print(f"  Sensitive nodes found: {result.sensitive_nodes_found}")
        print(f"  Total chains:          {result.total_chains}")
        print(f"  Max risk score:        {result.max_risk:.4f}")
        print(f"  Avg risk score:        {result.avg_risk:.4f}")
        
        if result.chains:
            print(f"\n  Detected proxy chains:")
            for chain, risk in zip(result.chains, result.risk_scores):
                path_str = " -> ".join(chain)
                risk_label = "HIGH" if risk >= 0.7 else "MEDIUM" if risk >= 0.3 else "LOW"
                print(f"    [{risk_label}] {path_str}  (risk={risk:.4f})")
        
        if result.recommendations:
            print(f"\n  Recommendations:")
            for rec in result.recommendations:
                print(f"    - {rec}")
                
    except Exception as e:
        print(f"Proxy chain detection failed: {e}")
else:
    print("Proxy chain detection skipped (requires DAG-GNN result with graph).")

In [ ]:
# Cell 8: Summary and conclusions
print("=" * 60)
print("Causal Discovery Experiment - Summary")
print("=" * 60)

print(f"""
Data:
  - {n_samples} samples, {n_features} features
  - Generated using sklearn.datasets.make_classification

Methods tested:
  - PC Algorithm:  {'completed' if pc_available else 'skipped'} ({pc_n_edges} edges)
  - DAG-GNN:       {'completed' if dag_result else 'skipped'} ({dag_n_edges} edges)

Key findings:
  - DAG-GNN learned a sparse graph (sparsity={dag_result.sparsity:.3f})
    if dag_result else 'N/A'
  - Proxy chains detected: {len(proxy_chains)}

Conclusions:
  1. DAG-GNN provides an end-to-end differentiable approach to causal discovery.
  2. The learned adjacency matrix reveals potential causal relationships.
  3. Proxy chain detection identifies paths where sensitive attributes may
     indirectly influence outcomes through non-sensitive variables.
  4. For production use, ensure sufficient data (1000+ samples) and
     increase training epochs for more stable results.
""")